# NumPy 파일별 2 · 06_B04 조건으로 고르고 축으로 요약하기

**한 줄 요약** · Boolean mask로 조건에 맞는 행 · 원소를 고르고, axis로 센서별 · 시점별 요약을 낼 때 결과 shape를 먼저 검증

- **원본** · `3. numpy/02_단계별실습/06_B04_boolean_axis_Student.ipynb` (원본 파일은 그대로 둠)
- **종합 정리본** · `00_최소선/03_NumPy` 전체 정리 4-1 · 4-2 · 6-4 (조건으로 꺼내기 · 계산 방향)

## 🎯 학습 목표

이 노트북을 끝내면 **자료를 보지 않고** 다음을 할 수 있어야 함

1. Python 숫자 · 0차원 배열 · NumPy scalar의 차이를 shape로 설명한다
2. 조건식으로 mask를 만들고, 조건 두 개는 괄호와 `&` · `|`로 묶는다
3. `(6,)` 행 mask와 `(6, 3)` 원소 mask의 결과 shape를 실행 전에 말한다
4. `axis=0` · `axis=1` · `keepdims=True`의 결과 shape를 고르고 센서 · 시점 요약에 쓴다

## 어디에 쓰나

*실제 상황 → 이 권에서 배우는 것*

- **센서 기록에서 조건에 맞는 시점만 남길 때** · 행 mask `raw[row_mask]` (2-1)
- **센서마다 기준값이 다른 경보 규칙을 만들 때** · 원소 mask · broadcasting (2-2 · 2-3)
- **특성(열)별 평균 · 표준편차로 정규화할 때** · `axis=0` · `keepdims=True` (3장)
- **조건 두 개를 묶다가 `ValueError`가 날 때** · `and` 대신 `&amp;` · 괄호 (1-3)

## 학습 지도

<img src="images/nb_map.png" alt="06_B04 학습 지도" width="763">

**예상 학습 시간** · 개념 50분 + 실습 20분

## 목차

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">장</th><th style="text-align: left;">제목</th><th style="text-align: left;">이 장의 질문</th></tr>
</thead>
<tbody>
<tr><td>0</td><td>준비</td><td>실행 환경 · 원본 문제 · 연습 데이터</td></tr>
<tr><td>1</td><td>배열 용어와 Boolean mask</td><td>숫자 하나 · mask · 조건 묶기는 각각 어떤 모양과 규칙을 가지는가?</td></tr>
<tr><td>2</td><td>행 mask와 원소 mask</td><td>mask의 모양에 따라 결과 shape는 어떻게 달라지는가?</td></tr>
<tr><td>3</td><td>axis로 요약하기</td><td>센서별 · 시점별 요약은 axis를 몇으로 하고, 결과 shape는?</td></tr>
<tr><td>4</td><td>종합 실습</td><td>조건에 맞는 행을 고르고 센서별 · 시점별 평균을 내려면?</td></tr>
<tr><td>—</td><td>마무리 · 부록</td><td>오늘 배운 것 · 기억 확인 · 오답 노트 · 면접 · 요약표 · 복습일 · 원본 대응표 · 전체 코드</td></tr>
</tbody></table>

---
## 0. 준비

원본과 같은 시점 × 센서 연습 데이터를 만듦. **셀은 위에서부터 차례로 실행**.

*항목 → 설정*

- **라이브러리** · `numpy`
- **seed · 장치** · 필요 없음 (무작위 · GPU 없음)
- **데이터** · `raw` (시점 6 × 센서 3) · 원본은 JSON 문자열로 넣었고 여기서는 같은 값을 배열로 직접 씀
- **축 의미** · 행 = 시점 · 열 = 센서(특성) · 원본은 열 0을 온도, 열 2를 압력으로 씀

**실행 흐름**

1. `numpy` 불러오기 · 버전 출력
2. 원본과 같은 센서 데이터 `raw` (6, 3) 만들기

In [1]:
# 실행 환경 확인 · 원본과 같은 연습 데이터 준비
# 왜 · 원본 문제와 같은 센서 데이터로 결과를 그대로 비교하기 위해
import numpy as np  # 배열 계산 라이브러리

print("NumPy:", np.__version__)            # 설치된 버전 기록
raw = np.array([[68, 0.31, 101.2],         # 시점 0: 센서 0 · 1 · 2
                [74, 0.48, 99.8],
                [76, 0.52, 103.1],
                [81, 0.61, 104.0],
                [72, 0.55, 98.9],
                [77, 0.44, 102.2]])
print("raw shape:", raw.shape, raw.dtype)  # (시점 6, 센서 3) · 실수

NumPy: 2.4.6
raw shape: (6, 3) float64


### 원본 문제

- **핵심 질문** · 조건에 맞는 행 또는 원소를 고르고 요약할 때 mask와 axis를 어떻게 검증할까?
- **실행 전 예측** · `(6, 3)` 데이터에 `(6,)` 행 mask와 `(6, 3)` 원소 mask를 적용했을 때 결과 shape
- **학생 문제** · 온도 78 이상이고 압력 103 이상인 행을 고르고 센서별 평균과 시점별 평균을 계산
  - mask와 결과 shape부터 제안
  - 출력은 `실제 값 / 결과 모양(shape) / 저장 형식(dtype) / 오류`로 기록
- **다른 조건에서 다시 확인** · 3행 2열 배열로 행 선택 · 원소 선택 · `keepdims` 비교

**원본 힌트**

- `(m,)` mask는 axis 0의 행을, `(m, n)` mask는 개별 원소를 고름
- 복합 배열 조건은 괄호와 `&` · `|`를 씀
- 기본 reduction(합계 · 평균)은 지정한 축을 없앰
- 축 번호만으로 센서의 실제 의미 · 단위까지 정해지지는 않음

---
## 1. 배열 용어와 Boolean mask

> **이 장의 흐름**
>
> - 원본 앞부분의 용어(scalar · Boolean · mask)를 코드로 확인
> - 조건 하나 → 조건 두 개(`&`) 순서로 mask를 만듦

<img src="images/nb_mask.png" alt="mask의 True 자리만 남고, 조건 두 개는 칸마다 &" width="775">

**파이프라인 위치** · ③ 전처리 — 조건으로 데이터 거르기

### 1-1. `B` 숫자 하나의 세 가지 모양 · `np.array(10)` · `np.float64(10)` <sub>원본 · 먼저 알아둘 배열 말</sub>

> 🎯 **`10` · `np.array(10)` · `np.float64(10)`은 무엇이 다른가?**

💡 **쉽게 말하면**

- 셋 다 숫자 10이지만 **담긴 그릇**이 다름.
- Python 숫자에는 shape가 없고, `np.array(10)`은 축이 0개인 배열.

**실행 흐름**

1. 세 표현 만들기
2. 타입 · 배열 관점 shape
3. hasattr · ndim 확인

**먼저 예측** · `np.asarray(10).shape`는? — 답은 코드 실행 결과 아래에

In [2]:
# 숫자 하나를 세 가지로 표현하기
# 왜 · 같은 10이라도 자료형에 따라 쓸 수 있는 속성이 다르다는 것을 보이기 위해
# 같은 10
values = {"Python 10": 10, "np.array(10)": np.array(10), "np.float64(10)": np.float64(10)}
for name, v in values.items():  # 표현마다
    # 자료형 · 배열 관점 모양
    print(f"{name:<15} 타입: {type(v).__name__:<8} shape: {np.asarray(v).shape}")


# Python 숫자에는 .shape가 없음
# 왜 · 배열로 바꿔야 shape · ndim을 쓸 수 있다는 것을 확인하기 위해
print("hasattr(10, 'shape'):", hasattr(10, "shape"))  # 속성이 있는가
print("np.array(10).ndim:", np.array(10).ndim)        # 0차원 배열

Python 10       타입: int      shape: ()
np.array(10)    타입: ndarray  shape: ()
np.float64(10)  타입: float64  shape: ()
hasattr(10, 'shape'): False
np.array(10).ndim: 0


**결과** · 예측 정답은 `()` — 축이 없는 0차원

- `Python 10       타입: int      shape: ()` — 배열로 보면 shape ()
- `np.array(10)    타입: ndarray  shape: ()` — 0차원 ndarray
- `np.float64(10)  타입: float64  shape: ()` — NumPy scalar · 별도 타입
- `hasattr(10, 'shape'): False` — Python 숫자 자체에는 `.shape`가 없음
- `np.array(10).ndim: 0` — 축 개수 0

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>Python <code>10</code></td><td><code>.shape</code> 없음</td><td>일반 계산</td></tr>
<tr><td><code>np.array(10)</code></td><td>shape <code>()</code> · 0차원 ndarray</td><td>배열 연산의 결과가 숫자 하나일 때</td></tr>
<tr><td><code>np.float64(10)</code></td><td>NumPy scalar</td><td><code>a.sum()</code> · <code>a[0]</code>처럼 배열에서 꺼낸 값</td></tr>
<tr><td><code>np.asarray(v).shape</code></td><td>무엇이든 배열 관점 모양</td><td>모양을 통일해 확인할 때</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 세 표현이 완전히 같지는 않음 → 모양이 중요하면 `np.asarray(v).shape`로 확인

> ✅ **한 줄 정리** · 숫자 하나도 Python 숫자 · 0차원 배열 · NumPy scalar로 다르며, 배열 관점 모양은 `np.asarray(v).shape`로 확인한다.

**딥러닝 코드에서는**

- `loss.item()` — 0차원 텐서를 Python 숫자로
- `acc = (pred == y).mean()` — 결과는 NumPy scalar

### 1-2. `A` mask로 True 자리 고르기 · `a[mask]` · `a > 15` <sub>원본 · 먼저 생각</sub>

> 🎯 **Boolean 배열을 대괄호에 넣으면 무엇이 남는가?**

💡 **쉽게 말하면**

- mask는 **O · X 출석부**.
- 같은 길이의 배열에 대면 O인 자리만 남음.

**실행 흐름**

1. mask 직접 적기
2. a[mask]
3. 조건식으로 mask 만들기

**먼저 예측** · `a[np.array([True, False, True])]`는? — 답은 코드 실행 결과 아래에

In [3]:
# Boolean 배열(mask)로 True 자리만 고르기
# 왜 · 위치 번호 대신 True/False 표로 값을 고르는 원리를 보기 위해
a = np.array([10, 20, 30])            # 값 3개
mask = np.array([True, False, True])  # 고를 자리 = True
print("a[mask] =", a[mask])           # 첫 · 셋째 값


# 조건식이 mask를 만들어 줌
# 왜 · mask를 손으로 쓰지 않고 조건식으로 만드는 방법을 익히기 위해
cond = a > 15                        # 칸마다 True/False
print("a > 15 →", cond, cond.dtype)  # bool 배열
print("a[a > 15] =", a[a > 15])      # 조건에 맞는 값만

a[mask] = [10 30]
a > 15 → [False  True  True] bool
a[a > 15] = [20 30]


**결과** · 예측 정답은 `[10 30]`

- `a[mask] = [10 30]` — True인 첫 · 셋째 값
- `a > 15 → [False  True  True] bool` — 조건식은 같은 모양의 bool 배열
- `a[a > 15] = [20 30]` — 조건에 맞는 값만

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>a[mask]</code></td><td>True 자리의 <strong>값</strong></td><td>조건에 맞는 데이터 꺼내기</td></tr>
<tr><td><code>a &gt; 15</code></td><td><strong>bool 배열</strong> (같은 shape)</td><td>조건을 먼저 눈으로 확인</td></tr>
<tr><td><code>np.where(mask)[0]</code></td><td>True의 <strong>위치</strong></td><td>몇 번째인지 필요할 때</td></tr>
<tr><td><code>mask.sum()</code></td><td>True <strong>개수</strong></td><td>조건에 맞는 개수</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - mask 길이가 고를 축 길이와 다르면 `IndexError`
> - 다른 배열에 쓸 땐 같은 순서인지 확인

> ✅ **한 줄 정리** · 조건식으로 만든 bool 배열(mask)을 대괄호에 넣으면 True 자리의 값만 남는다.

**딥러닝 코드에서는**

- `X[y == 1]` — 양성 샘플만
- `boxes[scores > 0.5]` — 점수가 높은 박스만

### 1-3. `A` 조건 두 개는 `&` · 괄호 · `|` <sub>원본 코드 셀 · 오류 예시</sub>

> 🎯 **조건 두 개를 묶을 때 왜 `and`가 아니라 `&`이고, 괄호는 왜 필요한가?**

💡 **쉽게 말하면**

- `and`는 **질문 하나에 예 · 아니오 하나**만 답함.
- 배열은 칸마다 답이 필요하므로 칸마다 계산하는 `&`를 씀.

**실행 흐름**

1. (열 0 >= 75) & (열 1 >= 0.5)
2. row_mask 확인
3. and로 바꿔 오류 확인

**먼저 예측** · `(raw[:, 0] >= 75) and (raw[:, 1] >= 0.5)`는 실행될까? — 답은 코드 실행 결과 아래에

In [4]:
# 조건 두 개: 열 0이 75 이상 그리고 열 1이 0.5 이상
# 왜 · 두 조건을 칸마다 동시에 따지려면 &가 필요하기 때문에
row_mask = (raw[:, 0] >= 75) & (raw[:, 1] >= 0.5)  # 칸마다 '그리고'
print("row_mask:", row_mask)                       # 시점마다 True/False


# and는 배열 전체를 참 · 거짓 하나로 정할 수 없어 오류
# 왜 · 원본의 오류 예시를 셀을 멈추지 않고 재현해 원인을 보기 위해
try:
    (raw[:, 0] >= 75) and (raw[:, 1] >= 0.5)  # 원본의 오류 예시
except ValueError as e:                       # 오류를 잡아 문장만 출력
    print("and → ValueError:", str(e)[:58])   # 앞부분만

row_mask: [False False  True  True False False]
and → ValueError: The truth value of an array with more than one element is 


**결과** · 예측 정답은 안 됨 — `ValueError`

- `row_mask: [False False  True  True False False]` — 시점 2 · 3만 두 조건 모두 만족
- `and → ValueError: The truth value of an array with more than one element is` — 배열 전체를 참 · 거짓 하나로 정할 수 없음

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>&amp;</code></td><td>칸마다 그리고</td><td>배열 조건 (기본 선택)</td></tr>
<tr><td><code>&#124;</code></td><td>칸마다 또는</td><td>둘 중 하나라도</td></tr>
<tr><td><code>~</code></td><td>칸마다 반대</td><td>조건 뒤집기</td></tr>
<tr><td><code>and</code> · <code>or</code></td><td>값 하나의 참 · 거짓만</td><td>배열에는 쓰지 않음</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 괄호를 빼면 `&`가 비교보다 먼저 계산되어 오류 또는 다른 뜻

> ✅ **한 줄 정리** · 배열 조건은 `&` · `|` · `~`로 칸마다 묶고, 조건 하나하나를 괄호로 감싼다.

**딥러닝 코드에서는**

- `valid = (x >= 0) & (x < W)` — 이미지 안쪽 좌표만
- Pandas도 `df[(df.a > 1) & (df.b < 5)]`

---
## 2. 행 mask와 원소 mask

> **이 장의 흐름**
>
> - 1장에서 mask와 `&`를 만듦
> - 이번에는 mask 모양이 `(6,)`일 때와 `(6, 3)`일 때 결과가 어떻게 다른지 비교

<img src="images/nb_rowelem.png" alt="행 mask는 행을 통째로, 원소 mask는 칸 하나씩 고름" width="758">

**파이프라인 위치** · ③ 전처리 · ⑥ 평가 — 조건 필터 · 경보 규칙

### 2-1. `A` 행 mask (6,) · `raw[row_mask]` <sub>원본 코드 셀 · 실행 전 예측</sub>

> 🎯 **`(6,)` 행 mask를 `(6, 3)` 데이터에 쓰면 결과 shape는?**

💡 **쉽게 말하면**

- 행 mask는 **시점마다 하나**씩 O · X.
- O인 시점의 센서 값 3개가 통째로 남음.

**실행 흐름**

1. row_mask shape 확인
2. raw[row_mask]
3. 결과 shape (k, 3)

**먼저 예측** · row_mask에 True가 2개면 `raw[row_mask].shape`는? — 답은 코드 실행 결과 아래에

In [5]:
# 행 mask (6,) → 조건에 맞는 시점(행)을 통째로 고름
# 왜 · mask 모양이 행 수와 같을 때 행 단위로 골라진다는 것을 확인하기 위해
print("row_mask shape:", row_mask.shape)            # 시점 수와 같은 길이
selected_rows = raw[row_mask]                       # True인 행만
print("raw[row_mask] shape:", selected_rows.shape)  # (고른 행 수, 센서 3)
print(selected_rows)                                # 고른 시점의 센서 값

row_mask shape: (6,)
raw[row_mask] shape: (2, 3)
[[ 76.     0.52 103.1 ]
 [ 81.     0.61 104.  ]]


**결과** · 예측 정답은 `(2, 3)` — 고른 행 2개 × 센서 3

- `row_mask shape: (6,)` — 시점 수와 같은 길이
- `raw[row_mask] shape: (2, 3)` — 행 2개를 통째로 · 열은 그대로 3

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>행 mask <code>(6,)</code></td><td>행을 통째로 → <code>(k, 3)</code></td><td>조건에 맞는 시점 · 샘플 고르기</td></tr>
<tr><td><code>raw[row_mask, 0]</code></td><td>고른 행의 한 열 → <code>(k,)</code></td><td>고른 시점의 한 센서만</td></tr>
<tr><td><code>raw[:, col_mask]</code></td><td>열 mask <code>(3,)</code> → 열을 고름</td><td>특성(센서) 고르기</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 행 mask는 축 0 길이와 같아야 함
> - 같은 mask로 다른 배열을 고를 땐 순서가 같은지 확인

> ✅ **한 줄 정리** · `(행 수,)` 모양의 mask는 행을 통째로 골라 `(고른 행 수, 열 수)`를 만든다.

**딥러닝 코드에서는**

- `X_pos = X[y == 1]` — 라벨 조건으로 샘플(행) 고르기

### 2-2. `A` 원소 mask (6, 3) · `raw > threshold` <sub>원본 코드 셀 · 실행 전 예측</sub>

> 🎯 **센서마다 기준이 다른 `(6, 3)` 원소 mask를 쓰면 결과 shape는?**

💡 **쉽게 말하면**

- 원소 mask는 **칸마다** O · X.
- O인 칸만 모으므로 행 · 열 구조가 사라지고 한 줄로 펴짐.

**실행 흐름**

1. 센서별 기준 (3,)
2. raw > threshold (broadcasting)
3. raw[element_mask]
4. 1차원 결과

**먼저 예측** · `raw[element_mask]`는 몇 차원일까? — 답은 코드 실행 결과 아래에

In [6]:
# 원소 mask (6, 3) → 센서마다 다른 기준으로 칸 하나씩 판단
# 왜 · 센서마다 기준이 다를 때 브로드캐스팅으로 한 번에 비교하기 위해
threshold = np.array([75, 0.5, 103])              # 센서 0 · 1 · 2의 기준값
element_mask = raw > threshold                    # (6, 3) vs (3,) → 행마다 같은 기준 (broadcasting)
print("element_mask shape:", element_mask.shape)  # 원본과 같은 모양
picked = raw[element_mask]                        # True인 칸만
print("raw[element_mask] shape:", picked.shape)   # 1차원으로 펴짐
print(picked)                                     # 어느 행 · 열이었는지 정보는 사라짐

element_mask shape: (6, 3)
raw[element_mask] shape: (8,)
[ 76.     0.52 103.1   81.     0.61 104.     0.55  77.  ]


**결과** · 예측 정답은 1차원 `(8,)` — True 칸 8개를 한 줄로

- `element_mask shape: (6, 3)` — (6, 3) vs (3,) → 행마다 같은 기준으로 비교
- `raw[element_mask] shape: (8,)` — True 칸 8개 · 행 · 열 정보는 사라짐
- `[ 76.     0.52 103.1   81.     0.61 104.     0.55  77.  ]` — 행 순서대로 읽은 값 (센서가 섞여 있음)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>원소 mask <code>(6, 3)</code></td><td>칸마다 → <strong>1차원</strong><code>(t,)</code></td><td>기준을 넘은 값만 모을 때</td></tr>
<tr><td>행 mask <code>(6,)</code></td><td>행 통째로 → <code>(k, 3)</code></td><td>시점 단위로 고를 때</td></tr>
<tr><td><code>raw &gt; threshold</code></td><td>(6, 3) vs (3,) broadcasting</td><td>센서마다 다른 기준</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 결과가 펴지면 어느 센서 값인지 알 수 없음 → 단위가 다른 값을 섞어 평균 내지 말 것

> ✅ **한 줄 정리** · 원소 mask는 칸마다 골라 결과를 1차원으로 펴므로, 위치 정보가 필요하면 다른 방법을 쓴다.

**딥러닝 코드에서는**

- `probs[probs > 0.5]` — 기준을 넘은 확률 값만 모으기

### 2-3. `B` 원소 mask의 위치 유지 · `mask.sum(axis=0)` · `np.where()` <sub>보충 · 원소 mask 확장</sub>

> 🎯 **원소 mask로 고르면서 어느 센서였는지 정보를 잃지 않으려면?**

💡 **쉽게 말하면**

- **세기** · 열마다 True 개수를 세면 센서별 결과가 남음.
- **가리기** · `np.where`로 거짓 칸만 빈칸(nan)으로 바꾸면 표 모양이 그대로.

**실행 흐름**

1. 열마다 True 개수
2. np.where로 거짓 칸 nan
3. 시점 3 확인

**먼저 예측** · `np.where(element_mask, raw, np.nan)`의 shape는? — 답은 코드 실행 결과 아래에

In [7]:
# 원소 mask를 버리지 않고 쓰는 두 방법
# 왜 · 원소 mask로 고르면 1차원으로 펴지는 문제를 개수 세기 · np.where로 피하기 위해
print("센서별 기준 초과 개수:", element_mask.sum(axis=0))  # True = 1로 열마다 셈
kept = np.where(element_mask, raw, np.nan)                 # 조건이 거짓인 칸만 nan
print("np.where 결과 shape:", kept.shape)                  # 위치를 유지한 (6, 3)
print("시점 3:", kept[3])                                  # 세 센서 모두 기준 초과

센서별 기준 초과 개수: [3 3 2]
np.where 결과 shape: (6, 3)
시점 3: [ 81.     0.61 104.  ]


**결과** · 예측 정답은 `(6, 3)` — 원본 위치 유지

- `센서별 기준 초과 개수: [3 3 2]` — 센서 0 · 1은 3번, 센서 2는 2번 초과
- `np.where 결과 shape: (6, 3)` — 펴지지 않고 표 모양 유지
- `시점 3: [ 81.     0.61 104.  ]` — 세 센서 모두 기준을 넘은 시점

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>raw[element_mask]</code></td><td>값만 1차원으로</td><td>넘은 값 목록</td></tr>
<tr><td><code>element_mask.sum(axis=0)</code></td><td>센서(열)별 개수</td><td>센서별 경보 횟수</td></tr>
<tr><td><code>np.where(mask, raw, np.nan)</code></td><td>위치 유지 · 거짓 칸 nan</td><td>표 모양을 지켜야 할 때</td></tr>
<tr><td><code>element_mask.any(axis=1)</code></td><td>시점마다 하나라도 넘었나</td><td>경보 시점 찾기</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - nan이 섞인 배열의 평균은 nan → `np.nanmean()`을 씀

> ✅ **한 줄 정리** · 원소 mask는 `sum(axis=0)`으로 세거나 `np.where`로 가려 위치 정보를 지킨다.

**딥러닝 코드에서는**

- `np.where(mask, x, 0)` — 패딩 위치를 0으로
- `mask.any(axis=1)` — 샘플마다 조건 확인

---
## 3. axis로 요약하기

> **이 장의 흐름**
>
> - 2장에서 고른 데이터를 이제 한 방향으로 줄여 요약
> - axis가 지정한 축이 사라진다는 규칙과 keepdims로 남기는 법

<img src="images/nb_axis.png" alt="axis=0은 센서마다, axis=1은 시점마다 결과 하나" width="758">

**파이프라인 위치** · ③ 전처리 — 특성별 평균 · 정규화

### 3-1. `A` 센서별 · 시점별 평균 · `mean(axis=0)` · `mean(axis=1)` <sub>원본 코드 셀 · 학생 문제</sub>

> 🎯 **`raw.mean(axis=0)`과 `raw.mean(axis=1)`의 shape는?**

💡 **쉽게 말하면**

- `axis=0`은 **세로로 접어** 열(센서)마다 하나.
- `axis=1`은 **가로로 접어** 행(시점)마다 하나.

**실행 흐름**

1. axis=0 센서별 평균
2. axis=1 시점별 평균
3. shape 비교

**먼저 예측** · `(6, 3)`에서 `mean(axis=0)`의 shape는? — 답은 코드 실행 결과 아래에

In [8]:
# axis=0 → 행(시점)을 따라 내려가며 계산 → 센서마다 결과 하나
# 왜 · 센서별 평균처럼 열마다 하나의 값을 얻기 위해
feature_mean = raw.mean(axis=0)                                       # 센서별 평균
print("센서별 평균:", np.round(feature_mean, 2), feature_mean.shape)  # (3,)


# axis=1 → 열(센서)을 따라 옆으로 계산 → 시점마다 결과 하나
# 왜 · axis=0과 결과 모양이 어떻게 다른지 비교하기 위해
row_mean = raw.mean(axis=1)  # 시점별 평균 (단위가 다른 센서를 섞음)
print("시점별 평균:", np.round(row_mean, 2), row_mean.shape)  # (6,)

센서별 평균: [ 74.67   0.48 101.53] (3,)
시점별 평균: [56.5  58.09 59.87 61.87 57.15 59.88] (6,)


**결과** · 예측 정답은 `(3,)` — 사라진 축은 0번(시점)

- `센서별 평균: [ 74.67   0.48 101.53] (3,)` — 센서 3개마다 평균 하나
- `시점별 평균: [56.5  58.09 59.87 61.87 57.15 59.88] (6,)` — 시점마다 하나 · 단위가 다른 센서를 섞은 값

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>axis=0</code></td><td>행 축이 사라짐 → <strong>열마다</strong></td><td>센서 · 특성별 통계 (정규화)</td></tr>
<tr><td><code>axis=1</code></td><td>열 축이 사라짐 → <strong>행마다</strong></td><td>시점 · 샘플별 요약</td></tr>
<tr><td>axis 생략</td><td>전체 하나</td><td>전체 합계 · 평균</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 시점별 평균은 온도 · 압력처럼 단위가 다른 값을 섞음 → 계산은 되지만 해석은 보류

> ✅ **한 줄 정리** · `axis`에 적은 축이 사라지므로 센서별 요약은 `axis=0`, 시점별 요약은 `axis=1`이다.

**딥러닝 코드에서는**

- `X.mean(axis=0)` · `X.std(axis=0)` — 특성별 정규화 통계
- `logits.argmax(axis=1)` — 샘플마다 예측

### 3-2. `A` keepdims로 축 남기기 · `keepdims=True` <sub>원본 · 다른 조건에서 다시 확인</sub>

> 🎯 **`keepdims=True`를 주면 결과 shape는 어떻게 달라지고, 어디에 쓰는가?**

💡 **쉽게 말하면**

- 접은 축을 버리지 않고 **두께 1로 남겨** 둠.
- 그래서 원본과 바로 빼기 · 나누기를 할 수 있음.

**실행 흐름**

1. 원본 retry 3행 2열
2. 행 선택 · 원소 선택
3. keepdims 비교
4. 평균 빼기

**먼저 예측** · `(3, 2)`에서 `mean(axis=0, keepdims=True)`의 shape는? — 답은 코드 실행 결과 아래에

In [9]:
# 원본 '다른 조건에서 다시 확인': keepdims로 사라질 축을 1로 남김
# 왜 · 축을 줄여도 원래 배열과 브로드캐스팅되도록 모양을 남기기 위해
retry = np.array([[1, 10], [2, 20], [3, 30]])                                       # 3행 2열
print("행 선택:", retry[retry[:, 0] >= 2].tolist(), retry[retry[:, 0] >= 2].shape)  # 행 mask
# 원소 mask
print("원소 선택:", retry[retry >= np.array([2, 20])], retry[retry >= np.array([2, 20])].shape)
# (2,) vs (1, 2)
print("keepdims 비교:", retry.mean(axis=0).shape, retry.mean(axis=0, keepdims=True).shape)


# keepdims 쓰임새: 센서마다 평균을 빼서 0 중심으로
# 왜 · 정규화처럼 열마다 평균을 빼는 전처리를 한 줄로 하기 위해
centered = raw - raw.mean(axis=0, keepdims=True)                          # (6, 3) - (1, 3)
print("빼고 난 센서별 평균 ≈ 0:", np.allclose(centered.mean(axis=0), 0))  # 아주 작은 오차만 남음

행 선택: [[2, 20], [3, 30]] (2, 2)
원소 선택: [ 2 20  3 30] (4,)
keepdims 비교: (2,) (1, 2)
빼고 난 센서별 평균 ≈ 0: True


**결과** · 예측 정답은 `(1, 2)`

- `행 선택: [[2, 20], [3, 30]] (2, 2)` — 열 0이 2 이상인 행 2개
- `원소 선택: [ 2 20  3 30] (4,)` — 열마다 기준 [2, 20] 이상인 칸 · 행 순서대로
- `keepdims 비교: (2,) (1, 2)` — 축을 버리면 (2,) · 남기면 (1, 2)
- `빼고 난 센서별 평균 ≈ 0: True` — 센서마다 평균을 빼 0 중심으로 (계산 오차는 아주 작음)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>mean(axis=0)</code></td><td><code>(3,)</code> · 축 사라짐</td><td>평균 값만 볼 때</td></tr>
<tr><td><code>mean(axis=0, keepdims=True)</code></td><td><code>(1, 3)</code> · 축 유지</td><td>원본에서 바로 빼거나 나눌 때</td></tr>
<tr><td><code>mean(axis=1, keepdims=True)</code></td><td><code>(6, 1)</code></td><td>행마다 평균으로 나눌 때 (이 경우 필수)</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - `axis=1` 결과로 원본을 나눌 때 keepdims가 없으면 `(6, 3)` vs `(6,)`라 오류

> ✅ **한 줄 정리** · `keepdims=True`는 사라질 축을 길이 1로 남겨, 요약값으로 원본을 바로 빼거나 나눌 수 있게 한다.

**딥러닝 코드에서는**

- `x - x.mean(axis=-1, keepdims=True)` — LayerNorm의 평균 빼기

---
## 4. 종합 실습 · 조건 행 고르고 요약하기

> **이 장의 흐름**
>
> - 1 · 2 · 3장을 원본 학생 문제에 그대로 적용
> - mask와 결과 shape를 먼저 적고 코드로 확인

<img src="images/nb_practice.png" alt="조건 행 2개를 고른 뒤 축 방향으로 요약" width="772">

**파이프라인 위치** · ③ 전처리 → ⑥ 평가

> 🎯 **온도 78 이상이고 압력 103 이상인 행의 센서별 평균과 시점별 평균은?**

💡 **쉽게 말하면**

- 조건 두 개 → `&`로 행 mask → 고른 행 → axis 방향만 정하면 끝.
- 코드 전에 mask shape `(4,)`, 결과 shape `(k, 3)`을 적어 둠.

**실행 흐름**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">단계</th><th style="text-align: left;">하는 일</th><th style="text-align: left;">쓰는 문법</th></tr>
</thead>
<tbody>
<tr><td>1</td><td>열 의미: 열 0 = 온도 · 열 2 = 압력</td><td>0장 · 1-2</td></tr>
<tr><td>2</td><td>조건 두 개를 괄호와 <code>&amp;</code>로</td><td>1-3</td></tr>
<tr><td>3</td><td>행 mask로 고르기 → <code>(k, 3)</code></td><td>2-1</td></tr>
<tr><td>4</td><td>센서별 평균 <code>axis=0</code> · 시점별 평균 <code>axis=1</code></td><td>3-1</td></tr>
<tr><td>5</td><td>값 / shape / dtype / 오류 기록</td><td>원본 기록 형식</td></tr>
</tbody></table>

**먼저 예측** · 조건을 만족하는 행은 몇 개이고 `feature_mean`의 shape는? — 답은 코드 실행 결과 아래에

In [10]:
# 원본 학생 문제: 온도(열 0) 78 이상 그리고 압력(열 2) 103 이상인 행
# 왜 · 원본 학생 문제를 그대로 풀어 정답 형식과 맞추기 위해
problem_raw = np.array([[70, .40, 100],                            # 시점 0
                        [76, .51, 103],
                        [79, .48, 104],
                        [82, .63, 105]])
row_mask = (problem_raw[:, 0] >= 78) & (problem_raw[:, 2] >= 103)  # 조건마다 괄호
selected = problem_raw[row_mask]                                   # 조건에 맞는 행
print("row_mask:", row_mask, "| selected shape:", selected.shape)  # (4,) · (2, 3)

row_mask: [False False  True  True] | selected shape: (2, 3)


**실행 흐름**

1. 고른 행의 센서별 평균(`axis=0`)
2. 시점별 평균(`axis=1`)
3. 원본 기록 형식으로 출력

In [11]:
# 센서별 평균(axis=0) · 시점별 평균(axis=1)
# 왜 · 고른 행으로 두 방향 평균을 모두 구해 axis 차이를 다시 확인하기 위해
feature_mean = selected.mean(axis=0)                       # 고른 행들의 센서별 평균
row_mean = selected.mean(axis=1)                           # 고른 행마다 평균
print("feature_mean:", np.round(feature_mean, 3), feature_mean.shape, feature_mean.dtype)  # (3,)
print("row_mean:", np.round(row_mean, 3), row_mean.shape)  # (2,)


# 원본 기록 형식: 실제 값 / 결과 모양 / 저장 형식 / 오류
# 왜 · 원본이 요구한 기록 형식에 맞춰 결과를 빠짐없이 남기기 위해
print(f"기록 → 값 {selected.tolist()} / shape {selected.shape} "  # 원본이 요구한 기록 형식
      f"/ dtype {selected.dtype} / 오류 없음")

feature_mean: [ 80.5     0.555 104.5  ] (3,) float64
row_mean: [61.16  62.543] (2,)
기록 → 값 [[79.0, 0.48, 104.0], [82.0, 0.63, 105.0]] / shape (2, 3) / dtype float64 / 오류 없음


**결과** · 예측 정답은 2개 (온도 79 · 82) · `(3,)`

- `row_mask: [False False  True  True] | selected shape: (2, 3)` — 시점 2 · 3이 두 조건 모두 만족
- `feature_mean: [ 80.5     0.555 104.5  ] (3,) float64` — 고른 두 행의 센서별 평균
- `row_mean: [61.16  62.543] (2,)` — 행마다 평균 (단위가 섞여 해석은 보류)
- `기록 → 값 [[79.0, 0.48, 104.0], [82.0, 0.63, 105.0]] / shape (2, 3) / dtype float64 / 오류 없음` — 원본이 요구한 기록 형식

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>행 mask + <code>axis=0</code></td><td>조건 시점의 센서별 요약</td><td><strong>이 문제의 답</strong></td></tr>
<tr><td>원소 mask</td><td>칸 단위 · 1차원으로 펴짐</td><td>조건 칸만 모을 때</td></tr>
<tr><td><code>and</code>로 조건 묶기</td><td>ValueError</td><td>쓰지 않음</td></tr>
<tr><td><code>axis=1</code> 평균</td><td>단위가 다른 센서를 섞음</td><td>계산은 되지만 해석 주의</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 조건에 맞는 행이 0개면 평균은 nan + 경고 → `selected.size`로 먼저 확인

> ✅ **한 줄 정리** · 조건은 괄호와 `&`로 행 mask를 만들고, 고른 행을 `axis=0`으로 센서별 · `axis=1`로 시점별 요약한다.

**딥러닝 코드에서는**

- `X[y == c].mean(axis=0)` — 클래스 c 샘플의 특성 평균 (클래스 중심)
- `mask.sum()`이 0인지 먼저 확인 → 빈 배치에서 손실이 nan이 되는 것을 막음

---
## 그래서 오늘 무엇을 배웠는가?

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">학습 목표</th><th style="text-align: left;">할 수 있게 된 것</th><th style="text-align: left;">확인 위치</th></tr>
</thead>
<tbody>
<tr><td>1. 숫자 하나의 모양</td><td><code>10</code> · <code>np.array(10)</code> · <code>np.float64(10)</code> 모두 배열 관점 shape <code>()</code></td><td>1-1</td></tr>
<tr><td>2. mask 만들기 · 묶기</td><td><code>a[a &gt; 15]</code> = <code>[20 30]</code>, <code>and</code>는 ValueError · <code>&amp;</code>는 칸마다</td><td>1-2 · 1-3</td></tr>
<tr><td>3. 행 vs 원소 mask</td><td><code>(6,)</code> → <code>(2, 3)</code>, <code>(6, 3)</code> → <code>(8,)</code></td><td>2-1 · 2-2</td></tr>
<tr><td>4. axis · keepdims</td><td><code>axis=0</code> → <code>(3,)</code>, <code>keepdims</code> → <code>(1, 3)</code>, 학생 문제 평균 <code>[80.5 0.555 104.5]</code></td><td>3장 · 4</td></tr>
</tbody></table>

### 주제별 한 줄 · 반드시 기억할 것

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">등급</th><th style="text-align: left;">주제</th><th style="text-align: left;">반드시 기억할 것</th><th style="text-align: left;">딥러닝 코드에서</th></tr>
</thead>
<tbody>
<tr><td><code>A</code></td><td>mask</td><td>조건 → bool 배열 → 대괄호</td><td><code>X[y == 1]</code></td></tr>
<tr><td><code>A</code></td><td><code>&amp;</code> · 괄호</td><td><code>and</code> 대신 칸마다</td><td><code>(x &gt;= 0) &amp; (x &lt; W)</code></td></tr>
<tr><td><code>A</code></td><td>행 vs 원소 mask</td><td><code>(k, 3)</code> vs 1차원</td><td>샘플 고르기 vs 값 모으기</td></tr>
<tr><td><code>A</code></td><td>axis</td><td>지정한 축이 사라짐</td><td><code>X.mean(axis=0)</code></td></tr>
<tr><td><code>A</code></td><td>keepdims</td><td>축을 1로 남겨 바로 계산</td><td><code>x - x.mean(-1, keepdims=True)</code></td></tr>
<tr><td><code>B</code></td><td>scalar 세 가지</td><td>배열 관점 shape <code>()</code></td><td><code>loss.item()</code></td></tr>
</tbody></table>

---
## 기억 확인 문제

위 내용을 **보지 않고** 답한 뒤, ✅ 한 줄 정리와 비교해 ○ (정확) · △ (일부) · × (모름)를 표시. △ · ×인 섹션만 다시 보기.

### 1. 질문에 답하기

*섹션 → 🎯 질문*

- **1-1** · `np.array(10)`의 shape와 ndim은?
- **1-2** · `a[mask]`는 무엇을 남기는가?
- **1-3** · 배열 조건을 `and`로 묶으면?
- **2-1** · `(6,)` 행 mask의 결과 shape는?
- **2-2** · `(6, 3)` 원소 mask의 결과 shape는?
- **2-3** · 원소 mask로 위치를 지키려면?
- **3-1** · 센서별 평균은 axis를 몇으로?
- **3-2** · `keepdims=True`는 왜 쓰는가?

#### 내 답 · 이 셀을 더블클릭해서 작성

**1-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

### 2. 코드로 답하기

아래 빈 셀에 직접 작성 후 실행.

1. `raw`에서 열 2(압력)가 102 이상인 행을 고르고 shape 출력
2. 그 행들의 센서별 최댓값(`max(axis=0)`)과 shape 출력

In [12]:
# 코드 문제 직접 작성

### 3. 변형 문제 · 조건을 바꿔 다시 풀기

1. 학생 문제를 '온도 78 이상 **또는** 압력 103 이상'으로 바꾸면 고른 행 수는?
2. `raw`를 시점마다 최댓값으로 나눠 0 ~ 1로 만들려면 `keepdims`가 왜 필요한가?
3. 원소 mask에서 시점마다 '하나라도 기준 초과'인 시점 번호를 구하려면?

In [13]:
# 변형 문제 직접 작성

### 4. 모범 답안 · 1 ~ 3을 다 푼 뒤에 보기

#### 질문 답

*섹션 → 답*

- **1-1** · `()` · 0
- **1-2** · mask가 True인 자리의 값
- **1-3** · ValueError · `&amp;`와 괄호를 씀
- **2-1** · `(고른 행 수, 3)`
- **2-2** · 1차원 `(True 개수,)`
- **2-3** · `mask.sum(axis=0)`으로 세거나 `np.where(mask, raw, np.nan)`
- **3-1** · `axis=0` → `(3,)`
- **3-2** · 축을 1로 남겨 원본과 바로 빼기 · 나누기

#### 코드 문제 답

```python
high = raw[raw[:, 2] >= 102]
print(high.shape)  # (3, 3)
print(high.max(axis=0), high.max(axis=0).shape)  # [ 81.     0.61 104.  ] (3,)
```

#### 변형 문제 답

*문제 → 답*

- **1** · 3개 (시점 1 · 2 · 3) — 압력 103인 시점 1이 새로 포함
- **2** · `raw.max(axis=1)`은 `(6,)`라 `(6, 3)`과 끝자리가 안 맞음 → `keepdims=True`로 `(6, 1)`
- **3** · `np.where(element_mask.any(axis=1))[0]` → `[2 3 4 5]`

---
## 오답 노트

틀린 문제는 정답을 베끼지 말고 **원인**을 적음. 다음 복습 때 이 표의 "재발 방지 질문"부터 스스로에게 물어보기.

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">틀린 내용</th><th style="text-align: left;">틀린 이유</th><th style="text-align: left;">올바른 원칙</th><th style="text-align: left;">재발 방지 질문</th></tr>
</thead>
<tbody>
<tr><td>(예) 원소 mask 결과를 <code>(k, 3)</code>이라고 답함</td><td>행 mask와 같은 줄 앎</td><td>원소 mask는 칸 단위 → 1차원</td><td>mask의 shape가 데이터와 같은가, 한 축 길이와 같은가?</td></tr>
<tr><td></td><td></td><td></td><td></td></tr>
<tr><td></td><td></td><td></td><td></td></tr>
</tbody></table>

---
## 면접 30초 답변

**Q1. Boolean mask로 데이터를 고를 때 결과 모양은 어떻게 정해지나요?**
> - mask가 한 축 길이와 같은 1차원이면 그 축의 행이나 열을 통째로 골라 나머지 축이 유지됩니다.
> - mask가 데이터와 같은 모양이면 True인 칸만 모아 1차원으로 펴집니다.
> - 그래서 코드 전에 mask의 shape와 결과 shape를 먼저 적어 둡니다.

**Q2. 배열 조건에 and 대신 &를 쓰는 이유는요?**
> - and는 값 하나의 참과 거짓만 판단할 수 있어서 원소가 여러 개인 배열에 쓰면 ValueError가 납니다.
> - &는 칸마다 그리고를 계산하고, 비교보다 먼저 계산되므로 조건마다 괄호를 칩니다.

**Q3. axis와 keepdims는 어떻게 고르나요?**
> - axis에 적은 축이 사라지므로 열마다 결과가 필요하면 0, 행마다 필요하면 1입니다.
> - 그 결과로 원본을 바로 빼거나 나눠야 하면 keepdims=True로 축을 길이 1로 남겨 broadcasting이 맞게 합니다.

---
## 최종 요약표

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>np.asarray()</code> · <code>.ndim</code></td><td>numpy · ndarray 속성</td><td>배열 관점 모양 · 축 개수</td><td>1-1 <code>np.asarray(v).shape</code></td></tr>
<tr><td><code>a[mask]</code></td><td>NumPy 인덱싱</td><td>True 자리 고르기</td><td>1-2 <code>a[a &gt; 15]</code> · 2-1 <code>raw[row_mask]</code></td></tr>
<tr><td><code>&amp;</code> · <code>&#124;</code> · <code>~</code></td><td>NumPy 연산자</td><td>칸마다 그리고 · 또는 · 반대</td><td>1-3 <code>(raw[:, 0] &gt;= 75) &amp; (raw[:, 1] &gt;= 0.5)</code></td></tr>
<tr><td><code>raw &gt; threshold</code></td><td>비교 + broadcasting</td><td>센서별 기준 원소 mask</td><td>2-2 <code>element_mask = raw &gt; threshold</code></td></tr>
<tr><td><code>.sum(axis=0)</code> · <code>.any(axis=1)</code></td><td>ndarray 메서드</td><td>True 개수 · 하나라도</td><td>2-3 <code>element_mask.sum(axis=0)</code></td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>np.where(cond, a, b)</code></td><td>numpy</td><td>조건에 따라 값 고르기 (위치 유지)</td><td>2-3 <code>np.where(element_mask, raw, np.nan)</code></td></tr>
<tr><td><code>.mean(axis=)</code></td><td>ndarray 메서드</td><td>축 방향 평균</td><td>3-1 <code>raw.mean(axis=0)</code> · 4 <code>selected.mean(axis=1)</code></td></tr>
<tr><td><code>keepdims=True</code></td><td>통계 함수 인자</td><td>사라질 축을 1로 유지</td><td>3-2 <code>raw.mean(axis=0, keepdims=True)</code></td></tr>
<tr><td><code>np.round()</code></td><td>numpy</td><td>보기 좋게 반올림</td><td>3-1 <code>np.round(feature_mean, 2)</code></td></tr>
</tbody></table>

### 할 것 · 하지 말 것

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">✅ 할 것</th><th style="text-align: left;">⚠️ 하지 말 것</th></tr>
</thead>
<tbody>
<tr><td>mask · 결과 shape를 먼저 적기</td><td>결과가 행인지 1차원인지 모르고 평균</td></tr>
<tr><td>조건마다 괄호 + <code>&amp;</code></td><td>배열에 <code>and</code> · <code>or</code></td></tr>
<tr><td>원소 mask 위치가 필요하면 <code>np.where</code></td><td>단위가 다른 값을 섞어 평균</td></tr>
<tr><td>요약값으로 나눌 땐 <code>keepdims=True</code></td><td><code>axis=1</code> 결과로 바로 나누기</td></tr>
</tbody></table>

### 헷갈리는 짝 한눈에 보기

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">짝</th><th style="text-align: left;">차이</th><th style="text-align: left;">결론</th></tr>
</thead>
<tbody>
<tr><td>행 mask vs 원소 mask</td><td><code>(k, 3)</code> vs 1차원</td><td>mask 모양으로 판단</td></tr>
<tr><td><code>&amp;</code> vs <code>and</code></td><td>칸마다 vs 값 하나</td><td>배열은 <code>&amp;</code></td></tr>
<tr><td><code>axis=0</code> vs <code>axis=1</code></td><td>열마다 vs 행마다</td><td>결과 개수로 판단</td></tr>
<tr><td><code>keepdims</code> 없음 vs 있음</td><td><code>(3,)</code> vs <code>(1, 3)</code></td><td>원본과 계산할 땐 있음</td></tr>
<tr><td><code>a[mask]</code> vs <code>np.where(mask, a, nan)</code></td><td>펴진 값 vs 위치 유지</td><td>위치가 필요하면 <code>where</code></td></tr>
</tbody></table>

---
## 다음 복습일

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">언제</th><th style="text-align: left;">할 일</th><th style="text-align: left;">걸리는 시간</th></tr>
</thead>
<tbody>
<tr><td>내일</td><td>기억 확인 문제 1을 자료 없이 다시 답하고 오답 노트 갱신</td><td>10분</td></tr>
<tr><td>3일 뒤</td><td>변형 문제 3개 다시 풀기 · 그림을 가리고 결과 shape 말하기</td><td>15분</td></tr>
<tr><td>7일 뒤</td><td>4장 종합 실습을 빈 셀에 직접 다시 짜기</td><td>20분</td></tr>
<tr><td>14일 뒤</td><td>오답 노트에 남은 문제만 · 전체 정리 4 · 6장 ✅ 줄만 읽기</td><td>10분</td></tr>
</tbody></table>

---
## 부록

### 이 노트북을 만든 방법

- 원본 · `06_B04_boolean_axis_Student.ipynb`의 용어 설명 · 예측 · 코드 셀 · 학생 문제 · 다시 확인
- 1장 · 용어(scalar · mask)를 실행 코드로 · 2장 · 원본 코드 셀을 행 · 원소 mask로 나누고 위치 유지 방법을 보충
- 3장 · 원본 '다른 조건에서 다시 확인'과 keepdims 쓰임새 · 오류 예시 스위치는 try/except로 바꿈
- 모든 코드는 실행한 출력과 함께 저장 · 형식 v2.1 (FORMAT_V2.md 1 ~ 7절)

### 읽는 법

*표시 → 의미*

- **`A` · `B`** · **A** 딥러닝 코드에 바로 나오는 필수 · **B** 알아두면 되는 것
- **🎯** · 이번 섹션에서 답해야 할 질문
- **💡** · 쉬운 설명 · 비유
- **⚠️** · 자주 하는 실수
- **✅** · 기억할 결론 = 🎯 질문의 답

- **섹션 앞부분** · 🎯 질문 → 💡 쉬운 설명 → 실행 흐름 → 먼저 예측 → 코드
- **섹션 뒷부분** · 결과(예측 정답 · 출력 줄) → 차이 · 언제 쓰나 → ⚠️ 주의 → ✅ 한 줄 정리
- **코드 주석** · 코드 위는 이 단계에서 하는 일, 코드 옆은 그 줄이 필요한 이유 (줄마다 · 열 맞춤)

### 원본 → 섹션 대응표

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">원본</th><th style="text-align: left;">내용</th><th style="text-align: left;">이 노트북</th></tr>
</thead>
<tbody>
<tr><td>먼저 알아둘 배열 말</td><td>ndarray · shape · ndim · axis · dtype · scalar · Boolean · mask</td><td>1-1 · 1-2</td></tr>
<tr><td>먼저 생각 · 개념</td><td>mask · feature · <code>&amp;</code> · <code>|</code> · reduction</td><td>1-2 · 1-3 · 3-1</td></tr>
<tr><td>실행 전 예측</td><td><code>(6,)</code> 행 mask · <code>(6, 3)</code> 원소 mask의 결과 shape</td><td>2-1 · 2-2</td></tr>
<tr><td>코드 셀</td><td>row_mask · element_mask · axis 평균 · and 오류 예시</td><td>1-3 · 2-1 · 2-2 · 3-1</td></tr>
<tr><td>학생 문제 (TODO)</td><td>온도 78 이상 &amp; 압력 103 이상 · 센서별 · 시점별 평균</td><td>4 종합 실습</td></tr>
<tr><td>다른 조건에서 교정 · 재시도</td><td>3행 2열 · 행 · 원소 선택 · keepdims</td><td>3-2</td></tr>
<tr><td>응용 · 자기 점검</td><td>단위 · 대상 차원 · 선택 의미 기록</td><td>주의 · 오답 노트</td></tr>
</tbody></table>

### 전체 코드

1 ~ 4장 코드를 장별 셀로 모음. **위에서부터 이 셀들만 실행해도 전체가 동작**하며, 각 블록 머리 주석의 번호가 섹션.

**실행 흐름**

1. 1-1 숫자 하나를 세 가지로 표현
2. 1-2 Boolean mask로 고르기
3. 1-3 조건 두 개 `&`

In [14]:
# ---------- 1-1 ----------
# 숫자 하나를 세 가지로 표현하기
# 왜 · 같은 10이라도 자료형에 따라 쓸 수 있는 속성이 다르다는 것을 보이기 위해
# 같은 10
values = {"Python 10": 10, "np.array(10)": np.array(10), "np.float64(10)": np.float64(10)}
for name, v in values.items():  # 표현마다
    # 자료형 · 배열 관점 모양
    print(f"{name:<15} 타입: {type(v).__name__:<8} shape: {np.asarray(v).shape}")


# Python 숫자에는 .shape가 없음
# 왜 · 배열로 바꿔야 shape · ndim을 쓸 수 있다는 것을 확인하기 위해
print("hasattr(10, 'shape'):", hasattr(10, "shape"))  # 속성이 있는가
print("np.array(10).ndim:", np.array(10).ndim)        # 0차원 배열


# ---------- 1-2 ----------
# Boolean 배열(mask)로 True 자리만 고르기
# 왜 · 위치 번호 대신 True/False 표로 값을 고르는 원리를 보기 위해
a = np.array([10, 20, 30])            # 값 3개
mask = np.array([True, False, True])  # 고를 자리 = True
print("a[mask] =", a[mask])           # 첫 · 셋째 값


# 조건식이 mask를 만들어 줌
# 왜 · mask를 손으로 쓰지 않고 조건식으로 만드는 방법을 익히기 위해
cond = a > 15                        # 칸마다 True/False
print("a > 15 →", cond, cond.dtype)  # bool 배열
print("a[a > 15] =", a[a > 15])      # 조건에 맞는 값만


# ---------- 1-3 ----------
# 조건 두 개: 열 0이 75 이상 그리고 열 1이 0.5 이상
# 왜 · 두 조건을 칸마다 동시에 따지려면 &가 필요하기 때문에
row_mask = (raw[:, 0] >= 75) & (raw[:, 1] >= 0.5)  # 칸마다 '그리고'
print("row_mask:", row_mask)                       # 시점마다 True/False


# and는 배열 전체를 참 · 거짓 하나로 정할 수 없어 오류
# 왜 · 원본의 오류 예시를 셀을 멈추지 않고 재현해 원인을 보기 위해
try:
    (raw[:, 0] >= 75) and (raw[:, 1] >= 0.5)  # 원본의 오류 예시
except ValueError as e:                       # 오류를 잡아 문장만 출력
    print("and → ValueError:", str(e)[:58])   # 앞부분만

Python 10       타입: int      shape: ()
np.array(10)    타입: ndarray  shape: ()
np.float64(10)  타입: float64  shape: ()
hasattr(10, 'shape'): False
np.array(10).ndim: 0
a[mask] = [10 30]
a > 15 → [False  True  True] bool
a[a > 15] = [20 30]
row_mask: [False False  True  True False False]
and → ValueError: The truth value of an array with more than one element is 


**실행 흐름**

1. 2-1 행 mask
2. 2-2 원소 mask
3. 2-3 원소 mask 활용 두 방법

In [15]:
# ---------- 2-1 ----------
# 행 mask (6,) → 조건에 맞는 시점(행)을 통째로 고름
# 왜 · mask 모양이 행 수와 같을 때 행 단위로 골라진다는 것을 확인하기 위해
print("row_mask shape:", row_mask.shape)            # 시점 수와 같은 길이
selected_rows = raw[row_mask]                       # True인 행만
print("raw[row_mask] shape:", selected_rows.shape)  # (고른 행 수, 센서 3)
print(selected_rows)                                # 고른 시점의 센서 값


# ---------- 2-2 ----------
# 원소 mask (6, 3) → 센서마다 다른 기준으로 칸 하나씩 판단
# 왜 · 센서마다 기준이 다를 때 브로드캐스팅으로 한 번에 비교하기 위해
threshold = np.array([75, 0.5, 103])              # 센서 0 · 1 · 2의 기준값
element_mask = raw > threshold                    # (6, 3) vs (3,) → 행마다 같은 기준 (broadcasting)
print("element_mask shape:", element_mask.shape)  # 원본과 같은 모양
picked = raw[element_mask]                        # True인 칸만
print("raw[element_mask] shape:", picked.shape)   # 1차원으로 펴짐
print(picked)                                     # 어느 행 · 열이었는지 정보는 사라짐


# ---------- 2-3 ----------
# 원소 mask를 버리지 않고 쓰는 두 방법
# 왜 · 원소 mask로 고르면 1차원으로 펴지는 문제를 개수 세기 · np.where로 피하기 위해
print("센서별 기준 초과 개수:", element_mask.sum(axis=0))  # True = 1로 열마다 셈
kept = np.where(element_mask, raw, np.nan)                 # 조건이 거짓인 칸만 nan
print("np.where 결과 shape:", kept.shape)                  # 위치를 유지한 (6, 3)
print("시점 3:", kept[3])                                  # 세 센서 모두 기준 초과

row_mask shape: (6,)
raw[row_mask] shape: (2, 3)
[[ 76.     0.52 103.1 ]
 [ 81.     0.61 104.  ]]
element_mask shape: (6, 3)
raw[element_mask] shape: (8,)
[ 76.     0.52 103.1   81.     0.61 104.     0.55  77.  ]
센서별 기준 초과 개수: [3 3 2]
np.where 결과 shape: (6, 3)
시점 3: [ 81.     0.61 104.  ]


**실행 흐름**

1. 3-1 `axis=0` · `axis=1`
2. 3-2 `keepdims`와 평균 빼기

In [16]:
# ---------- 3-1 ----------
# axis=0 → 행(시점)을 따라 내려가며 계산 → 센서마다 결과 하나
# 왜 · 센서별 평균처럼 열마다 하나의 값을 얻기 위해
feature_mean = raw.mean(axis=0)                                       # 센서별 평균
print("센서별 평균:", np.round(feature_mean, 2), feature_mean.shape)  # (3,)


# axis=1 → 열(센서)을 따라 옆으로 계산 → 시점마다 결과 하나
# 왜 · axis=0과 결과 모양이 어떻게 다른지 비교하기 위해
row_mean = raw.mean(axis=1)  # 시점별 평균 (단위가 다른 센서를 섞음)
print("시점별 평균:", np.round(row_mean, 2), row_mean.shape)  # (6,)


# ---------- 3-2 ----------
# 원본 '다른 조건에서 다시 확인': keepdims로 사라질 축을 1로 남김
# 왜 · 축을 줄여도 원래 배열과 브로드캐스팅되도록 모양을 남기기 위해
retry = np.array([[1, 10], [2, 20], [3, 30]])                                       # 3행 2열
print("행 선택:", retry[retry[:, 0] >= 2].tolist(), retry[retry[:, 0] >= 2].shape)  # 행 mask
# 원소 mask
print("원소 선택:", retry[retry >= np.array([2, 20])], retry[retry >= np.array([2, 20])].shape)
# (2,) vs (1, 2)
print("keepdims 비교:", retry.mean(axis=0).shape, retry.mean(axis=0, keepdims=True).shape)


# keepdims 쓰임새: 센서마다 평균을 빼서 0 중심으로
# 왜 · 정규화처럼 열마다 평균을 빼는 전처리를 한 줄로 하기 위해
centered = raw - raw.mean(axis=0, keepdims=True)                          # (6, 3) - (1, 3)
print("빼고 난 센서별 평균 ≈ 0:", np.allclose(centered.mean(axis=0), 0))  # 아주 작은 오차만 남음

센서별 평균: [ 74.67   0.48 101.53] (3,)
시점별 평균: [56.5  58.09 59.87 61.87 57.15 59.88] (6,)
행 선택: [[2, 20], [3, 30]] (2, 2)
원소 선택: [ 2 20  3 30] (4,)
keepdims 비교: (2,) (1, 2)
빼고 난 센서별 평균 ≈ 0: True


**실행 흐름**

1. 원본 학생 문제의 조건으로 행 고르기
2. 두 방향 평균
3. 기록 형식으로 출력

In [17]:
# ---------- 4. 종합 실습 ----------
# 원본 학생 문제: 온도(열 0) 78 이상 그리고 압력(열 2) 103 이상인 행
# 왜 · 원본 학생 문제를 그대로 풀어 정답 형식과 맞추기 위해
problem_raw = np.array([[70, .40, 100],                            # 시점 0
                        [76, .51, 103],
                        [79, .48, 104],
                        [82, .63, 105]])
row_mask = (problem_raw[:, 0] >= 78) & (problem_raw[:, 2] >= 103)  # 조건마다 괄호
selected = problem_raw[row_mask]                                   # 조건에 맞는 행
print("row_mask:", row_mask, "| selected shape:", selected.shape)  # (4,) · (2, 3)


# 센서별 평균(axis=0) · 시점별 평균(axis=1)
# 왜 · 고른 행으로 두 방향 평균을 모두 구해 axis 차이를 다시 확인하기 위해
feature_mean = selected.mean(axis=0)                       # 고른 행들의 센서별 평균
row_mean = selected.mean(axis=1)                           # 고른 행마다 평균
print("feature_mean:", np.round(feature_mean, 3), feature_mean.shape, feature_mean.dtype)  # (3,)
print("row_mean:", np.round(row_mean, 3), row_mean.shape)  # (2,)


# 원본 기록 형식: 실제 값 / 결과 모양 / 저장 형식 / 오류
# 왜 · 원본이 요구한 기록 형식에 맞춰 결과를 빠짐없이 남기기 위해
print(f"기록 → 값 {selected.tolist()} / shape {selected.shape} "  # 원본이 요구한 기록 형식
      f"/ dtype {selected.dtype} / 오류 없음")

row_mask: [False False  True  True] | selected shape: (2, 3)
feature_mean: [ 80.5     0.555 104.5  ] (3,) float64
row_mean: [61.16  62.543] (2,)
기록 → 값 [[79.0, 0.48, 104.0], [82.0, 0.63, 105.0]] / shape (2, 3) / dtype float64 / 오류 없음
